# 27b — Top 25 Priority Ward Maps with SDP Contest Borders (Fixed Ward Codes)

This notebook creates map assets for the newest **Top 25 High Confidence** and **Top 25 Medium Confidence** report tables.

Outputs:

- one full **North West region** map;
- one zoom-in map for each subregion:
  - Cheshire
  - Cumbria
  - Greater Manchester
  - Lancashire
  - Merseyside
- optional red borders for wards where the SDP has previously stood a candidate, using the matched SDP campaign-validation layer.

Default styling:

- **Green fill** = Top 25 High Confidence row
- **Orange fill** = Top 25 Medium Confidence row
- **Red border** = SDP previously contested ward
- **Light grey base** = all other North West wards

This is a presentation/report asset notebook. It does not alter the model.

**27b fix:** report-ready Top 25 tables sometimes omit `WD25CD`. This version hydrates ward codes by joining on `LAD25NM` + `WD25NM` using the full appendix/review files before mapping.

## 27.1 Setup

Expected project folders:

```text
Electoral_Tribes/
  data/
    processed/
      report_assets_pre_adam_v1/
      caveat_resolution_v2/
      sdp_campaign_validation_v2/
      priority_maps_v1/
    geography/
      boundaries/
        <WD25 boundary file>.gpkg/.shp/.geojson
```

The notebook searches recursively inside `data/processed` for the required CSVs, so it should tolerate your current folder layout.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

try:
    import geopandas as gpd
except Exception as e:
    raise ImportError("This notebook requires geopandas for map generation. Install geopandas in the current environment.") from e

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
BOUNDARY_DIR = GEOGRAPHY_DIR

OUTPUT_DIR = PROCESSED_DIR / "priority_maps_v1"
MAP_DIR = OUTPUT_DIR / "maps"
TABLE_DIR = OUTPUT_DIR / "tables"
MANIFEST_DIR = OUTPUT_DIR / "manifest"

for d in [OUTPUT_DIR, MAP_DIR, TABLE_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1


## 27.2 Configuration

Update these filenames only if your exported report tables use different names.

If `top_25_high_confidence_rows_report_table_v1.csv` or `top_25_medium_confidence_rows_report_table_v1.csv` are not found, the notebook falls back to `appendix_high_confidence_top100_v1.csv` and `appendix_medium_confidence_top100_v1.csv`, taking the first 25 rows by score.

In [2]:
# Primary report-table filenames.
HIGH_TOP25_FILENAME = "top_25_high_confidence_rows_report_table_v1.csv"
MEDIUM_TOP25_FILENAME = "top_25_medium_confidence_rows_report_table_v1.csv"

# Fallback appendix filenames.
HIGH_TOP100_FALLBACK = "appendix_high_confidence_top100_v1.csv"
MEDIUM_TOP100_FALLBACK = "appendix_medium_confidence_top100_v1.csv"

# North West reportable base file for all ward shapes in the region.
NW_BASE_FILENAMES = [
    "north_west_reportable_main_review_v2.csv",
    "north_west_revised_consolidated_review_v2.csv",
    "pre_adam_party_transition_diagnostics_all_v4.csv",
    "north_west_consolidated_target_review_v1.csv",
]

# SDP validation file. Used only for red borders.
SDP_PROFILE_FILENAMES = [
    "sdp_campaign_wards_profile_v2.csv",
    "sdp_campaign_wards_profile_v1.csv",
    "pre_adam_sdp_campaign_wards_profile_v1.csv",
]

# Manual override if auto boundary detection fails.
BOUNDARY_FILE_OVERRIDE = None  # Example: r"C:\\path\\to\\WD25_boundaries.gpkg"

# Main styling.
COLORS = {
    "Top 25 High Confidence": "#167A3A",       # green
    "Top 25 Medium Confidence": "#E87500",     # orange
    "Base": "#E6E8EA",
    "Boundary": "#FFFFFF",
    "LADBoundary": "#5A5A5A",
    "SDPBorder": "#C00000",
    "Title": "#112A46",
}

# North West LAD groupings used for zoom maps.
SUBREGION_LADS = {
    "Cheshire": [
        "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington"
    ],
    "Cumbria": [
        "Cumberland", "Westmorland and Furness"
    ],
    "Greater Manchester": [
        "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford", "Stockport", "Tameside", "Trafford", "Wigan"
    ],
    "Lancashire": [
        "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde", "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley", "Rossendale", "South Ribble", "West Lancashire", "Wyre"
    ],
    "Merseyside": [
        "Knowsley", "Liverpool", "Sefton", "St. Helens", "St Helens", "Wirral"
    ],
}

# Plot toggles.
DRAW_SDP_CONTESTED_RED_BORDERS = True
DRAW_LAD_OUTLINES = True
FIGSIZE_REGION = (9, 13)
FIGSIZE_SUBREGION = (8, 9)
DPI = 260

manifest_rows = []

def add_manifest(filename, asset_type, geography, description, path):
    manifest_rows.append({
        "filename": filename,
        "asset_type": asset_type,
        "geography": geography,
        "description": description,
        "path": str(path),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })

## 27.3 Helper functions

In [3]:
def slugify(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def find_file(filename, required=True):
    direct = [
        PROCESSED_DIR / filename,
        OUTPUT_DIR / filename,
        NOTEBOOK_DIR / filename,
        PROJECT_DIR / filename,
        Path("/mnt/data") / filename,
    ]
    for p in direct:
        if p.exists():
            return p
    if PROCESSED_DIR.exists():
        matches = list(PROCESSED_DIR.rglob(filename))
        if matches:
            return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def read_csv_file(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional missing:", filename)
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def find_first_available(filenames, required=True):
    for fn in filenames:
        df, path = read_csv_file(fn, required=False)
        if df is not None:
            return df, path, fn
    if required:
        raise FileNotFoundError(f"Could not find any of: {filenames}")
    return None, None, None


def standardise_code(series):
    return series.astype("string").str.strip()


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def score_sort_col(df):
    candidates = [
        "initial_watchlist_score", "top_model_score", "model_score", "structural_opportunity_score",
        "score", "overall_score"
    ]
    return first_existing_col(df, candidates)


def take_top25(df):
    df = df.copy()
    sort_col = score_sort_col(df)
    if sort_col:
        df[sort_col] = pd.to_numeric(df[sort_col], errors="coerce")
        df = df.sort_values(sort_col, ascending=False)
    return df.head(25).copy()


def get_ward_code_col(df):
    candidates = ["WD25CD", "WD25CD_model", "WD25CD_final", "ward_code", "source_geography_code"]
    col = first_existing_col(df, candidates)
    if col is None:
        raise KeyError(f"No ward-code column found. Columns available: {df.columns.tolist()}")
    return col


def get_lad_name_col(df):
    candidates = ["LAD25NM", "LAD25NM_model", "lad_name", "council_name"]
    return first_existing_col(df, candidates)


def find_boundary_file():
    if BOUNDARY_FILE_OVERRIDE:
        p = Path(BOUNDARY_FILE_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(f"BOUNDARY_FILE_OVERRIDE does not exist: {p}")
        return p

    search_dirs = [BOUNDARY_DIR, GEOGRAPHY_DIR, PROCESSED_DIR, Path("/mnt/data")]
    patterns = ["*.gpkg", "*.shp", "*.geojson", "*.json"]
    candidates = []
    for folder in search_dirs:
        if folder.exists():
            for pattern in patterns:
                candidates.extend(folder.rglob(pattern))
    if not candidates:
        return None

    # Prefer WD25 / ward boundary files, but avoid obvious lookup tables.
    preferred = []
    for p in candidates:
        name = p.name.lower()
        if any(x in name for x in ["wd25", "ward"]):
            if not any(x in name for x in ["lookup", "lu", "to_lad", "to_ward"]):
                preferred.append(p)
    return preferred[0] if preferred else candidates[0]


def normalise_name_for_join(series):
    return (
        series.astype("string")
        .fillna("")
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )


def build_ward_code_reference():
    """Build a robust LAD/ward-name -> WD25CD reference from files that retain ward codes."""
    ref_files = [
        "north_west_reportable_main_review_v2.csv",
        "north_west_revised_consolidated_review_v2.csv",
        "appendix_high_confidence_top100_v1.csv",
        "appendix_medium_confidence_top100_v1.csv",
        "pre_adam_party_transition_diagnostics_all_v4.csv",
        "north_west_consolidated_target_review_v1.csv",
    ]
    frames = []
    for fn in ref_files:
        df, path = read_csv_file(fn, required=False)
        if df is None:
            continue
        code_col = first_existing_col(df, ["WD25CD", "WD25CD_model", "WD25CD_final", "ward_code", "source_geography_code"])
        lad_col = first_existing_col(df, ["LAD25NM", "LAD25NM_model", "lad_name", "council_name"])
        ward_col = first_existing_col(df, ["WD25NM", "WD25NM_model", "ward_name", "source_geography_name"])
        if code_col and lad_col and ward_col:
            tmp = df[[code_col, lad_col, ward_col]].copy()
            tmp.columns = ["WD25CD", "LAD25NM", "WD25NM"]
            tmp["source_reference_file"] = fn
            frames.append(tmp)
    if not frames:
        raise FileNotFoundError("Could not build a ward-code reference. No reference file with WD25CD/LAD25NM/WD25NM was found.")
    ref = pd.concat(frames, ignore_index=True)
    ref["WD25CD"] = standardise_code(ref["WD25CD"])
    ref["join_lad"] = normalise_name_for_join(ref["LAD25NM"])
    ref["join_ward"] = normalise_name_for_join(ref["WD25NM"])
    ref = ref.dropna(subset=["WD25CD"])
    ref = ref[ref["WD25CD"].ne("")]
    # If duplicates exist, keep the first. Reference order above prioritises full corrected review files.
    ref = ref.drop_duplicates(["join_lad", "join_ward"], keep="first")
    print("Ward-code reference rows:", len(ref))
    return ref


def ensure_wd25_codes(df, label):
    """Ensure a dataframe has WD25CD, using existing code column or LAD+ward name lookup."""
    df = df.copy()
    code_col = first_existing_col(df, ["WD25CD", "WD25CD_model", "WD25CD_final", "ward_code", "source_geography_code"])
    if code_col is not None:
        df["WD25CD"] = standardise_code(df[code_col])
        print(f"{label}: using existing ward-code column `{code_col}`.")
        return df

    lad_col = first_existing_col(df, ["LAD25NM", "LAD25NM_model", "lad_name", "council_name"])
    ward_col = first_existing_col(df, ["WD25NM", "WD25NM_model", "ward_name", "source_geography_name"])
    if lad_col is None or ward_col is None:
        raise KeyError(
            f"{label}: no WD25CD and no usable LAD/ward name columns. Columns available: {df.columns.tolist()}"
        )

    ref = build_ward_code_reference()
    df["join_lad"] = normalise_name_for_join(df[lad_col])
    df["join_ward"] = normalise_name_for_join(df[ward_col])
    before_cols = list(df.columns)
    df = df.merge(
        ref[["join_lad", "join_ward", "WD25CD", "source_reference_file"]],
        on=["join_lad", "join_ward"],
        how="left",
        validate="many_to_one"
    )
    missing = int(df["WD25CD"].isna().sum())
    print(f"{label}: hydrated WD25CD from LAD+ward names. Missing after lookup: {missing} of {len(df)}")
    if missing:
        display_cols = [c for c in [lad_col, ward_col, "initial_watchlist_score", "revised_strategic_lane_v2"] if c in df.columns]
        print(f"{label}: unmatched rows for manual inspection:")
        display(df.loc[df["WD25CD"].isna(), display_cols].head(25))
        raise KeyError(f"{label}: {missing} rows still missing WD25CD after LAD+ward lookup. Add WD25CD manually or update reference files.")
    df = df.drop(columns=["join_lad", "join_ward"], errors="ignore")
    return df


## 27.4 Load priority tables

This loads the Top 25 high/medium confidence lists, with fallback to the Top 100 appendix files if needed.

In [4]:

high, high_path = read_csv_file(HIGH_TOP25_FILENAME, required=False)
if high is None:
    high, high_path = read_csv_file(HIGH_TOP100_FALLBACK, required=True)
    high = take_top25(high)
    print("Using fallback high-confidence Top 100 file and taking top 25.")
else:
    high = take_top25(high)

medium, medium_path = read_csv_file(MEDIUM_TOP25_FILENAME, required=False)
if medium is None:
    medium, medium_path = read_csv_file(MEDIUM_TOP100_FALLBACK, required=True)
    medium = take_top25(medium)
    print("Using fallback medium-confidence Top 100 file and taking top 25.")
else:
    medium = take_top25(medium)

# Report-ready Top 25 tables often omit WD25CD for readability.
# 27b hydrates WD25CD from full appendix/review files using LAD25NM + WD25NM.
high = ensure_wd25_codes(high, "High-confidence Top 25")
medium = ensure_wd25_codes(medium, "Medium-confidence Top 25")

high["WD25CD"] = standardise_code(high["WD25CD"])
medium["WD25CD"] = standardise_code(medium["WD25CD"])

priority = pd.concat([
    high.assign(priority_map_category="Top 25 High Confidence"),
    medium.assign(priority_map_category="Top 25 Medium Confidence"),
], ignore_index=True, sort=False)

# Resolve duplicates defensively. High confidence wins if a row appears in both.
priority["priority_rank"] = priority["priority_map_category"].map({
    "Top 25 High Confidence": 1,
    "Top 25 Medium Confidence": 2,
})
priority = priority.sort_values("priority_rank").drop_duplicates("WD25CD", keep="first")

print("High confidence Top 25 rows:", len(high))
print("Medium confidence Top 25 rows:", len(medium))
print("Unique priority wards:", priority["WD25CD"].nunique())
priority[["WD25CD", "priority_map_category"]].head()


Loaded top_25_high_confidence_rows_report_table_v1.csv: (25, 12) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\top_25_high_confidence_rows_report_table_v1.csv
Loaded top_25_medium_confidence_rows_report_table_v1.csv: (25, 14) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\top_25_medium_confidence_rows_report_table_v1.csv
Loaded north_west_reportable_main_review_v2.csv: (824, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_reportable_main_review_v2.csv
Loaded north_west_revised_consolidated_review_v2.csv: (825, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_revised_consolidated_review_v2.csv
Loaded appendix_high_confidence_top100_v1.csv: (671, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_high_confidence_top100_v1.csv
Loaded ap

,WD25CD,priority_map_category
0,E05014894,Top 25 High Confidence
1,E05015206,Top 25 High Confidence
2,E05014652,Top 25 High Confidence
3,E05014823,Top 25 High Confidence
4,E05015188,Top 25 High Confidence


## 27.5 Load North West base and SDP contested wards

The North West base gives the full region outline and LAD names. The SDP profile is only used for red borders.

In [5]:
nw_base, nw_base_path, nw_base_name = find_first_available(NW_BASE_FILENAMES, required=True)
nw_base_code_col = get_ward_code_col(nw_base)
nw_base["WD25CD"] = standardise_code(nw_base[nw_base_code_col])

lad_col = get_lad_name_col(nw_base)
if lad_col is None:
    raise KeyError("Could not identify an LAD/council name column in the North West base file.")

nw_base["LAD25NM_base"] = nw_base[lad_col].astype("string").fillna("").str.strip()
nw_codes = set(nw_base["WD25CD"].dropna())

sdp, sdp_path, sdp_name = find_first_available(SDP_PROFILE_FILENAMES, required=False)
if sdp is not None:
    sdp_code_col = first_existing_col(sdp, ["WD25CD_model", "WD25CD_final", "WD25CD", "ward_code"])
    if sdp_code_col is None:
        print("SDP profile found but no usable WD25 code column was identified. Red borders disabled.")
        sdp_codes = set()
    else:
        sdp["WD25CD_sdp"] = standardise_code(sdp[sdp_code_col])
        sdp_codes = set(sdp["WD25CD_sdp"].dropna()) & nw_codes
        print("Matched SDP-contested North West WD25 wards:", len(sdp_codes))
else:
    sdp_codes = set()
    print("No SDP profile found. Red borders disabled.")

print("North West base rows:", len(nw_base))
print("North West unique WD25 codes:", len(nw_codes))

Loaded north_west_reportable_main_review_v2.csv: (824, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_reportable_main_review_v2.csv
Loaded sdp_campaign_wards_profile_v2.csv: (171, 64) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2\sdp_campaign_wards_profile_v2.csv
Matched SDP-contested North West WD25 wards: 6
North West base rows: 824
North West unique WD25 codes: 824


## 27.6 Load ward boundaries and build mapping layer

The boundary file must contain ward geometries and a ward code column equivalent to `WD25CD`.

In [6]:
boundary_file = find_boundary_file()
if boundary_file is None:
    raise FileNotFoundError("No boundary file found. Put a WD25 boundary .gpkg/.shp/.geojson file in data/geography/boundaries.")

print("Boundary file selected:", boundary_file)
wards = gpd.read_file(boundary_file)
print("Boundary rows:", len(wards))
print("Boundary columns:", wards.columns.tolist())

# Standardise WD25CD column.
if "WD25CD" not in wards.columns:
    wd_candidates = [c for c in wards.columns if c.upper() == "WD25CD" or "WD25CD" in c.upper()]
    if not wd_candidates:
        wd_candidates = [c for c in wards.columns if c.upper().startswith("WD") and c.upper().endswith("CD")]
    if not wd_candidates:
        raise KeyError("Could not identify WD25CD in boundary file.")
    wards = wards.rename(columns={wd_candidates[0]: "WD25CD"})

wards["WD25CD"] = standardise_code(wards["WD25CD"])

# Merge North West base first to restrict boundary layer to North West reportable wards.
nw_map = wards.merge(
    nw_base[["WD25CD", "LAD25NM_base"]].drop_duplicates("WD25CD"),
    on="WD25CD",
    how="inner"
)

# Merge priority flags.
priority_small = priority[["WD25CD", "priority_map_category"]].drop_duplicates("WD25CD")
nw_map = nw_map.merge(priority_small, on="WD25CD", how="left")
nw_map["priority_map_category"] = nw_map["priority_map_category"].fillna("Not in Top 25")
nw_map["is_priority_top25"] = nw_map["priority_map_category"].ne("Not in Top 25")
nw_map["sdp_contested"] = nw_map["WD25CD"].isin(sdp_codes)

# Reproject for sensible map rendering.
try:
    nw_map = nw_map.to_crs(27700)
except Exception:
    print("Could not reproject to EPSG:27700; using source CRS.")

print("North West map rows:", len(nw_map))
print(nw_map["priority_map_category"].value_counts(dropna=False))
print("SDP-contested wards visible in North West map:", int(nw_map["sdp_contested"].sum()))

Boundary file selected: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg
Boundary rows: 8405
Boundary columns: ['WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'LAD25NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']
North West map rows: 824
priority_map_category
Not in Top 25               774
Top 25 Medium Confidence     25
Top 25 High Confidence       25
Name: count, dtype: int64
SDP-contested wards visible in North West map: 6


## 27.7 Map plotting functions

In [12]:
def dissolve_lad_boundaries(gdf):
    if not DRAW_LAD_OUTLINES:
        return None
    try:
        return gdf.dissolve(by="LAD25NM_base", as_index=False)
    except Exception:
        return None


def plot_priority_map(gdf, geography_name, filename, figsize=(9, 13)):
    if gdf.empty:
        print(f"Skipping {geography_name}: no rows")
        return None

    fig, ax = plt.subplots(figsize=figsize)

    # Base layer: all wards.
    gdf.plot(
        ax=ax,
        color=COLORS["Base"],
        edgecolor=COLORS["Boundary"],
        linewidth=0.12,
        zorder=1,
    )

    # Priority fills.
    for category in ["Top 25 High Confidence", "Top 25 Medium Confidence"]:
        sub = gdf[gdf["priority_map_category"].eq(category)]
        if len(sub) > 0:
            sub.plot(
                ax=ax,
                color=COLORS[category],
                edgecolor="white",
                linewidth=0.22,
                zorder=3,
            )

    # LAD outlines for context.
    lad_outline = dissolve_lad_boundaries(gdf)
    if lad_outline is not None and len(lad_outline) > 0:
        lad_outline.boundary.plot(
            ax=ax,
            color=COLORS["LADBoundary"],
            linewidth=0.45,
            alpha=0.55,
            zorder=4,
        )

    # SDP contested red borders.
    if DRAW_SDP_CONTESTED_RED_BORDERS:
        sdp_sub = gdf[gdf["sdp_contested"]]
        if len(sdp_sub) > 0:
            sdp_sub.boundary.plot(
                ax=ax,
                color=COLORS["SDPBorder"],
                linewidth=1.05,
                zorder=6,
            )

    title = f"{geography_name}: Top 50 scoring wards"
    ax.set_title(title, fontsize=18, color=COLORS["Title"], weight="bold", pad=16)
    ax.set_axis_off()

    legend_elements = [
        Patch(facecolor=COLORS["Top 25 High Confidence"], edgecolor="white", label="Top 25 High Confidence"),
        Patch(facecolor=COLORS["Top 25 Medium Confidence"], edgecolor="white", label="Top 25 Medium Confidence"),
        Patch(facecolor=COLORS["Base"], edgecolor="white", label="Other North West wards"),
    ]
    if DRAW_SDP_CONTESTED_RED_BORDERS:
        legend_elements.append(Line2D([0], [0], color=COLORS["SDPBorder"], lw=2, label="SDP previously contested"))
    if DRAW_LAD_OUTLINES:
        legend_elements.append(Line2D([0], [0], color=COLORS["LADBoundary"], lw=1, label="Council boundary"))

    ax.legend(handles=legend_elements, loc="lower left", frameon=True, fontsize=8)

    out_path = MAP_DIR / filename
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    add_manifest(filename, "map_png", geography_name, "Top 25 high/medium confidence priority wards with SDP-contested red borders.", out_path)
    print("Saved:", out_path)
    return out_path

## 27.8 Generate North West region map

In [13]:
region_map_path = plot_priority_map(
    nw_map,
    "North West",
    "map_27_north_west_top25_high_medium_sdp_borders_v1.png",
    figsize=FIGSIZE_REGION,
)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_north_west_top25_high_medium_sdp_borders_v1.png


## 27.9 Generate subregion zoom maps

These maps use the LAD groupings defined in section 27.2.

In [14]:
subregion_paths = []
for subregion, lads in SUBREGION_LADS.items():
    lads_clean = {str(x).strip().lower() for x in lads}
    sub = nw_map[nw_map["LAD25NM_base"].astype(str).str.strip().str.lower().isin(lads_clean)].copy()

    filename = f"map_27_{slugify(subregion)}_top25_high_medium_sdp_borders_v1.png"
    path = plot_priority_map(sub, subregion, filename, figsize=FIGSIZE_SUBREGION)
    if path is not None:
        subregion_paths.append(path)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_cheshire_top25_high_medium_sdp_borders_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_cumbria_top25_high_medium_sdp_borders_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_greater_manchester_top25_high_medium_sdp_borders_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_lancashire_top25_high_medium_sdp_borders_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\maps\map_27_merseyside_top25_high_medium_sdp_borders_v1.png


## 27.10 Export map index and summary tables

In [10]:
# Ward-level map index.
map_index_cols = ["WD25CD", "LAD25NM_base", "priority_map_category", "sdp_contested"]
map_index = nw_map[map_index_cols].copy()
map_index["map_priority_label"] = np.where(
    map_index["priority_map_category"].eq("Not in Top 25"),
    "Base ward",
    map_index["priority_map_category"],
)

map_index_path = TABLE_DIR / "map_27_priority_ward_index_v1.csv"
map_index.to_csv(map_index_path, index=False)
add_manifest(map_index_path.name, "table_csv", "All maps", "Ward-level index used for priority maps.", map_index_path)

# Summary by subregion.
summary_rows = []
for subregion, lads in SUBREGION_LADS.items():
    lads_clean = {str(x).strip().lower() for x in lads}
    sub = nw_map[nw_map["LAD25NM_base"].astype(str).str.strip().str.lower().isin(lads_clean)].copy()
    summary_rows.append({
        "subregion": subregion,
        "wards_total": int(len(sub)),
        "top25_high_confidence_wards": int(sub["priority_map_category"].eq("Top 25 High Confidence").sum()),
        "top25_medium_confidence_wards": int(sub["priority_map_category"].eq("Top 25 Medium Confidence").sum()),
        "sdp_contested_wards_visible": int(sub["sdp_contested"].sum()),
    })

subregion_summary = pd.DataFrame(summary_rows)
summary_path = TABLE_DIR / "map_27_subregion_priority_summary_v1.csv"
subregion_summary.to_csv(summary_path, index=False)
add_manifest(summary_path.name, "table_csv", "Subregion summaries", "Counts of priority and SDP-contested wards by subregion.", summary_path)

display(subregion_summary)

,subregion,wards_total,top25_high_confidence_wards,top25_medium_confidence_wards,sdp_contested_wards_visible
0,Cheshire,137,5,0,1
1,Cumbria,79,2,0,0
2,Greater Manchester,215,7,0,3
3,Lancashire,252,6,24,1
4,Merseyside,141,5,1,1


## 27.11 Manifest

In [11]:
manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "map_27_priority_maps_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)
print("Manifest saved:", manifest_path)
display(manifest)

Manifest saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\priority_maps_v1\manifest\map_27_priority_maps_manifest_v1.csv


,filename,asset_type,geography,description,path,created_at
0,map_27_north_west_top25_high_medium_sdp_border...,map_png,North West,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:42
1,map_27_cheshire_top25_high_medium_sdp_borders_...,map_png,Cheshire,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:43
2,map_27_cumbria_top25_high_medium_sdp_borders_v...,map_png,Cumbria,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:45
3,map_27_greater_manchester_top25_high_medium_sd...,map_png,Greater Manchester,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:46
4,map_27_lancashire_top25_high_medium_sdp_border...,map_png,Lancashire,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:48
5,map_27_merseyside_top25_high_medium_sdp_border...,map_png,Merseyside,Top 25 high/medium confidence priority wards w...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:49
6,map_27_priority_ward_index_v1.csv,table_csv,All maps,Ward-level index used for priority maps.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:49
7,map_27_subregion_priority_summary_v1.csv,table_csv,Subregion summaries,Counts of priority and SDP-contested wards by ...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T17:39:49


## 27b.12 Notes for interpretation

These maps should be described as **report-priority review maps**, not final target maps.

Suggested caption language:

> Green wards are the current Top 25 high-confidence report-priority wards. Orange wards are the current Top 25 medium-confidence report-priority wards. Red borders indicate wards where the SDP has previously stood a candidate in the matched SDP validation dataset. These maps show structural review priority, not vote-share forecasts or confirmed target selections.